## Startup order — `depends_on` + healthchecks

`depends_on: [db]` does **less** than everyone assumes, and it's the single most common Compose bug.

**The trap.** By default, `depends_on` waits only for the container to have **started** — not for the service *inside* it to be **ready**. Compose starts `db` before `web`, yes, but Postgres's process exists after a millisecond while the server isn't accepting connections for another second or two. So `web` starts, tries to connect, and **crashes before `db` is ready**. "Started" is not "ready."

**The fix** is a **healthcheck** on the dependency plus a **condition** on `depends_on`:

```yaml
services:
  db:
    image: postgres:16
    environment:
      POSTGRES_PASSWORD: secret
    healthcheck:
      test: ["CMD-SHELL", "pg_isready -U postgres"]
      interval: 5s
      timeout: 3s
      retries: 5
      start_period: 10s

  web:
    image: myorg/web
    depends_on:
      db:
        condition: service_healthy
```

Now Compose starts `db`, **polls its healthcheck**, and only starts `web` once the check reports `healthy`. The conditions you'll use:

- **`service_started`** — the old default: wait for container start only.
- **`service_healthy`** — wait for the healthcheck to pass. *This is the one you want* for databases and APIs.
- **`service_completed_successfully`** — wait for a one-shot service (a migration container) to exit `0` — the clean way to run migrations before the app.

The healthcheck itself is worth understanding: `test` is the probe command (exit 0 = healthy), `interval`/`timeout`/`retries` govern polling, and **`start_period`** is a grace window where failures don't count against `retries` — essential for services that are slow to warm up. Get this pattern right once and flaky "connection refused on startup" bugs disappear.